# 終末地尋訪機率與風險分析：示範

<a href="https://colab.research.google.com/github/Isongzhe/arknights-endfield-gacha/blob/main/examples/demo.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

這份筆記示範怎麼用 `gacha` 套件，從你目前的狀態精確計算《明日方舟：終末地》的尋訪機率。
所有數字都由有限狀態的遞迴直接算出，不是模擬。規則以亞服為準，來源與確認狀態見 repo 的 `docs/assumptions.md`。

依序執行每一格即可。想算自己的情況，改掉標了「改這裡」的數字。

In [ ]:
# 安裝（在 Colab 上大約一分鐘）
%pip install -q "git+https://github.com/Isongzhe/arknights-endfield-gacha.git"

## 一、單一卡池：從目前的狀態算起

狀態用四個數字描述：保底計數 `t`（距離上次出六星已抽幾抽）、本池已抽 `n`、已拿到的 UP 張數 `c`、是否已從本池拿過 UP `u`。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from gacharisk.kernel.chain import EnumeratedChain
from gacharisk.kernel.forward import hitting_time
from gacharisk.models.endfield import BannerState, SingleBannerModel
from gacharisk.risk import metrics as rk
from gacharisk.rules.endfield import BannerSpec, EndfieldCharacterRules

# 改這裡：限定池，開頭 10 抽免費（登入 5 + 商店 5），保底計數 48，本池還沒抽
rules = EndfieldCharacterRules(free_start_pulls=10)
start = BannerState(t=48, n=0, c=0, u=0)
stock = 60  # 手上的抽數

model = SingleBannerModel(BannerSpec(rules, target_copies=1, cap=120), start=start)
dist = hitting_time(EnumeratedChain.from_model(model))

print(f"平均要自備      {rk.mean(dist):.1f} 抽")
print(f"中位數          {rk.quantile(dist, 0.5)} 抽")
print(f"九成情況內      {rk.quantile(dist, 0.9)} 抽")
print(f"最壞            {dist.horizon} 抽")
print(f"{stock} 抽內拿到    {rk.completion(dist, stock):.1%}")

In [ ]:
pmf = dist.f_succ
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].step(range(len(pmf)), pmf, where="mid")
axes[0].set(title="P(got it on exactly this pull)", xlabel="own pulls", ylabel="probability")
axes[1].step(range(len(pmf)), np.cumsum(pmf), where="post")
axes[1].axvline(stock, linestyle="--", color="gray")
axes[1].set(title="P(got it within this many pulls)", xlabel="own pulls", ylabel="cumulative probability")
plt.tight_layout(); plt.show()

分布有兩個尖峰：一個是必出六星的那一抽，一個是 120 抽的大保底。所以平均數沒有什麼代表性，多數人不是很快拿到，就是一路抽到大保底。

## 二、兩個池共用一筆抽數：復刻池要抽到哪裡停

復刻池與限定池的保底互不相通，只共用抽數。下面列出在復刻池每一個停手點，對兩邊機率的影響。

In [ ]:
from gacharisk.analysis.two_banner import cap_table, first_up_pmf, recommend
from gacharisk.rules.endfield import rerun_rules

# 改這裡：復刻池累積已抽 30、保底計數 30；限定池同上一節；到第二個池結束共有 89 抽
rerun = SingleBannerModel(BannerSpec(rerun_rules(), 1, 120), start=BannerState(30, 30, 0, 0))
limited = SingleBannerModel(BannerSpec(rules, 1, 120), start=BannerState(48, 0, 0, 0))
total_stock = 89

pmf_a, pmf_b = first_up_pmf(rerun), first_up_pmf(limited)
rows = cap_table(pmf_a, pmf_b, total_stock)

print("復刻池上限   復刻角色   新角色   兩隻都有")
for cap in (0, 20, 40, 50, 60, 70, total_stock):
    r = rows[cap]
    print(f"{cap:>8}    {r.p_first:>7.1%}  {r.p_second:>7.1%}  {r.p_both:>7.1%}")
print("建議上限:", recommend(rows).cap)

In [ ]:
caps = [r.cap for r in rows]
plt.figure(figsize=(7, 3.8))
plt.plot(caps, [r.p_both for r in rows], label="both")
plt.plot(caps, [r.p_first for r in rows], label="re-run target")
plt.plot(caps, [r.p_second for r in rows], label="next limited target")
plt.xlabel("cap on the re-run banner (own pulls)"); plt.ylabel("probability"); plt.legend(); plt.show()

機率不是平滑變化的。復刻角色的機率在軟保底區陡升，新角色的機率在「剩下的抽數不夠走到它的保底」時整段掉下去。所以合理的停手點只會落在這些門檻上。

## 三、缺的抽數要花多少錢

價目是資料，不是規則。預設是台幣、首儲雙倍已用完的價目。

In [ ]:
from gacharisk.cost.menu import ENDFIELD_TW_STANDARD as menu

need = (len(pmf_a) - 1) + (len(pmf_b) - 1)   # 兩隻都保證最多要幾抽
short = max(0, need - total_stock)
print(f"兩隻都保證最多要 {need} 抽，還差 {short} 抽，最便宜的補法是 NT${menu.min_cost(short):,}")
for k in (10, 30, 60, 100):
    print(f"補 {k:>3} 抽：NT${menu.min_cost(k):,}")

## 四、連續多個限定池

限定池之間保底會繼承，抽滿 60 抽還會送下一池 10 抽。所以第一池抽完會留東西給第二池，兩池合計需要的抽數，比「每池各自從零開始算再相加」來得少。下面把兩種算法並排比較。

In [ ]:
from gacharisk.models.plan import Plan

fresh = EndfieldCharacterRules()
want = BannerSpec(fresh, target_copies=1, cap=120)
skip = BannerSpec(fresh, target_copies=0, cap=0)

single = hitting_time(EnumeratedChain.from_model(SingleBannerModel(want)))
for name, banners in (("抽、抽", [want, want]), ("抽、跳、抽", [want, skip, want])):
    plan = hitting_time(EnumeratedChain.from_model(Plan(banners)))
    naive = rk.from_pmf(rk.convolve(single.pmf_stop, 2))
    print(f"{name}: 把跨池繼承算進去，平均要 {rk.mean(plan):.1f} 抽；"
          f"忽略繼承、每池都從零算，平均要 {rk.mean(naive):.1f} 抽。150 抽內全拿到 {rk.completion(plan, 150):.1%}")

## 五、武器池

武器池只用武庫配額，1,980 換一次十連。計算以十連為單位。

In [ ]:
from gacharisk.models.weapon import WeaponBannerModel
from gacharisk.rules.weapon import WeaponBannerRules

w = hitting_time(EnumeratedChain.from_model(WeaponBannerModel(WeaponBannerRules())))
cdf = np.cumsum(w.f_succ)
quota = 9000  # 改這裡：你的武庫配額
can = quota // 1980
for k in range(1, 9):
    mark = "  <- 你的配額" if k == min(can, 8) else ""
    print(f"抽到第 {k} 次十連時已拿到 UP 武器：{cdf[k]:6.1%}{mark}")

## 六、驗證

遊戲內公告的六星綜合機率是「拿到 UP 之後 2.2720%」。用規則直接算：一個保底循環的平均長度，加上每 240 抽一個信物。

In [ ]:
p = np.array(EndfieldCharacterRules().probs)
cycle = np.concatenate([[1.0], np.cumprod(1 - p[:-1])]).sum()
print(f"保底循環平均 {cycle:.4f} 抽")
print(f"六星每抽 {1/cycle:.4%} + 信物每抽 {1/240:.4%} = {1/cycle + 1/240:.4%}（官方 2.2720%）")

## 接下來

- 命令列：`gacha-risk evaluate`、`gacha-risk decide`、`gacha-risk weapon`，用法見 README。
- 規則與來源：`docs/assumptions.md` 與 `docs/rules/`。
- 這是非官方的研究與同好專案。模型告訴你機率，不會改變機率；抽之前先決定自己的花費上限。